In [0]:
%pip install -q torch transformers safetensors huggingface_hub mlflow

In [0]:
dbutils.library.restartPython()

In [0]:
import re
import torch
from transformers import AutoModel, AutoTokenizer

# pre trained model loading
MODEL_ID = "fromziro/JetonCount"
REVISION = "b07f42658cdbeca99a34323c5d55f5d53df6cfcb9" 
TOKENIZER_ID = "fromziro/Er-Tiny-1.3M"

model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, revision=REVISION)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)

#the features we need
PUNCT = set(r""".,!?;:'"`~@#$%^&*()-_=+[]{}<>/\|""")
SYMB  = set(r"""@#$%^&*()-_=+[]{}<>/\|~`""")

def base_features(text: str, vocab_size: int):
    chars = len(text)
    words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    n = len(words)
    avg = sum(len(w) for w in words) / n if n else 0.0
    longest = max((len(w) for w in words), default=0)
    p = sum(c in PUNCT for c in text) / chars if chars else 0.0
    s = sum(c in SYMB for c in text) / chars if chars else 0.0
    return [float(chars), float(n), avg, p, s, float(longest), float(vocab_size)]

In [0]:
TARGET_VOCAB_SIZE = 199998 
MIN_CHARS_FOR_MODEL = 400         # roughly 70-80 words; below this we use heuristic

def predict_tokens(text, vocab_size=TARGET_VOCAB_SIZE):
    """Estimate token count. Returns the estimate, which method produced it, and debug info."""
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if text.strip() == "":
        return {"estimated_tokens": 0, "method": "empty", "approximate": False,
                "raw_prediction": None, "features": None}

    heuristic = max(1, round(len(text) / 4))

    if len(text) < MIN_CHARS_FOR_MODEL:
        return {"estimated_tokens": heuristic, "method": "heuristic_chars_div_4",
                "approximate": True, "raw_prediction": None, "features": None}

    feats = base_features(text, vocab_size)
    x = torch.tensor([feats], dtype=torch.float32)     
    with torch.inference_mode():
        raw = model(input_features=x).logits.item()

    if raw <= 0:    #making sure token amount is not negative                               
        return {"estimated_tokens": heuristic, "method": "heuristic_fallback",
                "approximate": True, "raw_prediction": raw, "features": None}

    return {"estimated_tokens": round(raw), "method": "jetoncount", "approximate": False,
            "raw_prediction": raw, "features": dict(zip(model.config.base_feature_names, feats))}

# quick test generated by claude
long_text = ("The city council met on Tuesday to discuss the new transit plan, which would add three bus lines "
             "and extend service hours into the late evening for residents on the east side. Researchers found "
             "that regular exercise improved memory in older adults, although the effect depended on how often "
             "participants trained and how long the study lasted. When the storm finally passed, the harbor was "
             "quiet again, and the fishermen began untangling their nets while gulls circled overhead looking for "
             "scraps. A good budget starts with knowing where your money goes each month.")
for label, t in [("empty", ""), ("short", "Hello world"), ("sentence", "Can you help me write an email to my landlord?"), ("long", long_text)]:
    r = predict_tokens(t)
    print(f"{label:<9} est={r['estimated_tokens']:>4}  method={r['method']:<22} approximate={r['approximate']}")

In [0]:

import mlflow.pyfunc
import pandas as pd
import os
from huggingface_hub import snapshot_download

class JetonCountPyfunc(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        import torch
        from transformers import AutoModel

        model_dir = context.artifacts["model_dir"]

        self.model = AutoModel.from_pretrained(
            model_dir,
            trust_remote_code=True
        )
        self.model.eval()

    def predict(self, context, model_input, params=None):
        results = []

        for text in model_input["text"]:
            try:
                if not isinstance(text, str):
                    raise TypeError("text must be a string")

                # Use the existing hybrid prediction function
                result = predict_tokens(text)

                results.append({
                    "estimated_tokens": float(result["estimated_tokens"]),
                    "method": result["method"],
                    "approximate": result["approximate"],
                    "error": None
                })

            except Exception as e:
                results.append({
                    "estimated_tokens": None,
                    "method": "error",
                    "approximate": True,
                    "error": str(e)
                })

        return pd.DataFrame(results)


In [0]:

import mlflow
import mlflow.pyfunc
import pandas as pd
import os
import shutil
from huggingface_hub import snapshot_download

MODEL_ID = "fromziro/JetonCount"
REVISION = "b07f42658cdbeca9934323c5d55f5d53df6cfcb9"

source_dir = snapshot_download(
    repo_id=MODEL_ID,
    revision=REVISION,
    allow_patterns=["*.py", "*.json", "*.safetensors"]
)

LOCAL_DIR = "/tmp/jetoncount_model"

if os.path.exists(LOCAL_DIR):
    shutil.rmtree(LOCAL_DIR)

shutil.copytree(source_dir, LOCAL_DIR, symlinks=False)


print("Model directory:", LOCAL_DIR)
print("Files:", os.listdir(LOCAL_DIR))
print("Model file exists:", os.path.isfile(
    os.path.join(LOCAL_DIR, "model.safetensors")
))

mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "workspace.default.jetoncount_estimator"

# Packaging the prediction function and model files
with mlflow.start_run(run_name="jetoncount_estimator"):
    info = mlflow.pyfunc.log_model(
        name="jetoncount",
        python_model=JetonCountPyfunc(),
        artifacts={"model_dir": LOCAL_DIR},
        pip_requirements=[
            "torch==2.15.0",
            "transformers==5.22.2",
            "safetensors",
            "pandas",
        ],
        input_example=pd.DataFrame({
            "text": ["Hello world"]
        }),
        registered_model_name=MODEL_NAME,
    )

print("Registered:", MODEL_NAME)
print("Model URI:", info.model_uri)

# Testing the packaged model
loaded = mlflow.pyfunc.load_model(info.model_uri)

result = loaded.predict(pd.DataFrame({
    "text": ["Hello world", long_text]
}))

print(result.to_string())
